In [ ]:
!pip install bpemb

In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from bpemb import BPEmb
from torch.nn import CrossEntropyLoss
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn import preprocessing
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset
from transformers import Trainer
from tqdm.auto import tqdm

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Only with the three languages
df_lang_train = df_train[df_train['language'].isin(['ar', 'ko', 'te'])]
df_lang_val = df_val[df_val['language'].isin(['ar', 'ko', 'te'])]


### Functions from week 36 and 37

In [ ]:
## Remove unwanted characters from the questions

def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    # pattern_context = re.compile(r"[?؟,;\/\\\[\]#():.]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

df_train_clean = cleanDf(df_train)
df_val_clean = cleanDf(df_val)

In [ ]:
arabicDf_train_clean = df_train_clean[df_train_clean['lang'] == 'ar'].copy()
teluguDf_train_clean = df_train_clean[df_train_clean['lang'] == 'te'].copy()
koreanDf_train_clean = df_train_clean[df_train_clean['lang'] == 'ko'].copy()

arabicDf_val_clean = df_val_clean[df_val_clean['lang'] == 'ar'].copy()
teluguDf_val_clean = df_val_clean[df_val_clean['lang'] == 'te'].copy()
koreanDf_val_clean = df_val_clean[df_val_clean['lang'] == 'ko'].copy()

### Week 38

### BiLSTM

In [ ]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load multilingual mBART50 model
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Map sprog -> mBART50 koder
LANG_MAP = {
    "te": "te_IN",   # Telugu
    "ar": "ar_AR",   # Arabisk
    "ko": "ko_KR"    # Koreansk
}
TARGET_LANG = "en_XX"  # Engelsk

# Oversættelsesfunktion
def translate_to_en_mbart(text, lang):
    if lang not in LANG_MAP:
        return text  # returner original hvis sprog ikke understøttet

    tokenizer.src_lang = LANG_MAP[lang]
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TARGET_LANG]
    )
    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_text

# Sortér datasættet, så Telugu kommer først
df_trans = df_train_clean[df_train_clean['lang'].isin(["te", "ar", "ko"])].reset_index(drop=True)
df_trans['lang'] = pd.Categorical(df_trans['lang'], categories=["te", "ar", "ko"], ordered=True)
df_trans = df_trans.sort_values('lang').reset_index(drop=True)

# Oversæt med tqdm

df_trans_ar = df_trans[df_trans['lang'] == "ar"].head(300)
df_trans_ko = df_trans[df_trans['lang'] == "ko"].head(300)
df_trans_te = df_trans[df_trans['lang'] == "te"].head(300)

df_trans = pd.concat([df_trans_te, df_trans_ar, df_trans_ko], ignore_index=True)

tqdm.pandas(desc="Oversætter spørgsmål")
df_trans['question_en'] = df_trans.progress_apply(
    lambda row: translate_to_en_mbart(row['question'], row['lang']),
    axis=1
)


# Print 5 eksempler per sprog
for lang in ["te", "ar", "ko"]:
    print(f"\n{lang.upper()} eksempel:")
    print(df_trans[df_trans['lang']==lang]['question_en'].head(5).tolist())


In [ ]:
# Filtrer de ønskede sprog i validation-sættet
df_val_trans = df_val_clean[df_val_clean['lang'].isin(["te", "ar", "ko"])].reset_index(drop=True)

# Sortér sprog rækkefølge, så Telugu kommer først
df_val_trans['lang'] = pd.Categorical(df_val_trans['lang'], categories=["te", "ar", "ko"], ordered=True)
df_val_trans = df_val_trans.sort_values('lang').reset_index(drop=True)

# Tag de første 30 rækker fra hvert sprog
df_val_ar = df_val_trans[df_val_trans['lang'] == "ar"].head(100)
df_val_ko = df_val_trans[df_val_trans['lang'] == "ko"].head(100)
df_val_te = df_val_trans[df_val_trans['lang'] == "te"].head(100)

# Concatenate: Telugu først
df_val_trans = pd.concat([df_val_te, df_val_ar, df_val_ko], ignore_index=True)

# Oversæt med tqdm
tqdm.pandas(desc="Oversætter validation spørgsmål")
df_val_trans['question_en'] = df_val_trans.progress_apply(
    lambda row: translate_to_en_mbart(row['question'], row['lang']),
    axis=1
)

# Print 5 eksempler per sprog
for lang in ["te", "ar", "ko"]:
    print(f"\n{lang.upper()} eksempel:")
    print(df_val_trans[df_val_trans['lang']==lang]['question_en'].head(5).tolist())


In [ ]:
SEP = " [SEP] "

# Lav ny kolonne med question + SEP + context
df_trans['text'] = df_trans['question_en'] + SEP + df_trans['context']
df_val_trans['text'] = df_val_trans['question_en'] + SEP + df_val_trans['context']


# Arabisk
df_ar_train = df_trans[df_trans['lang'] == 'ar'].reset_index(drop=True)
df_ar_val = df_val_trans[df_val_trans['lang'] == 'ar'].reset_index(drop=True)

# Telugu
df_te_train = df_trans[df_trans['lang'] == 'te'].reset_index(drop=True)
df_te_val = df_val_trans[df_val_trans['lang'] == 'te'].reset_index(drop=True)

# Koreansk
df_ko_train = df_trans[df_trans['lang'] == 'ko'].reset_index(drop=True)
df_ko_val = df_val_trans[df_val_trans['lang'] == 'ko'].reset_index(drop=True)


In [ ]:
df_lang_train['label'] = df_lang_train['answerable'].astype(int)
df_lang_val['label'] = df_lang_val['answerable'].astype(int)

In [ ]:
class BiLSTMNetwork(nn.Module):
    """
    Basic BiLSTM network
    """
    def __init__(
            self,
            pretrained_embeddings: torch.tensor,
            lstm_dim: int,
            dropout_prob: float = 0.1,
            n_classes: int = 2
    ):
        """
        Initializer for basic BiLSTM network
        :param pretrained_embeddings: A tensor containing the pretrained BPE embeddings
        :param lstm_dim: The dimensionality of the BiLSTM network
        :param dropout_prob: Dropout probability
        :param n_classes: The number of output classes
        """

        # First thing is to call the superclass initializer
        super(BiLSTMNetwork, self).__init__()

        # We'll define the network in a ModuleDict, which makes organizing the model a bit nicer
        # The components are an embedding layer, a 2 layer BiLSTM, and a feed-forward output layer
        self.model = nn.ModuleDict({
            'embeddings': nn.Embedding.from_pretrained(pretrained_embeddings, padding_idx=pretrained_embeddings.shape[0] - 1),
            'bilstm': nn.LSTM(
                pretrained_embeddings.shape[1],
                lstm_dim,
                1,
                batch_first=True,
                dropout=dropout_prob,
                bidirectional=True),
            'cls': nn.Linear(2*lstm_dim, n_classes)
        })
        self.n_classes = n_classes
        self.dropout = nn.Dropout(p=dropout_prob)

        # Initialize the weights of the model
        self._init_weights()

    def _init_weights(self):
        all_params = list(self.model['bilstm'].named_parameters()) + \
                     list(self.model['cls'].named_parameters())
        for n,p in all_params:
            if 'weight' in n:
                nn.init.xavier_normal_(p)
            elif 'bias' in n:
                nn.init.zeros_(p)

    def forward(self, inputs, input_lens, labels = None):
        """
        Defines how tensors flow through the model
        :param inputs: (b x sl) The IDs into the vocabulary of the input samples
        :param input_lens: (b) The length of each input sequence
        :param labels: (b) The label of each sample
        :return: (loss, logits) if `labels` is not None, otherwise just (logits,)
        """

        # Get embeddings (b x sl x edim)
        embeds = self.model['embeddings'](inputs)

        # Pack padded: This is necessary for padded batches input to an RNN
        lstm_in = nn.utils.rnn.pack_padded_sequence(
            embeds,
            input_lens.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # Pass the packed sequence through the BiLSTM
        lstm_out, hidden = self.model['bilstm'](lstm_in)

        # Unpack the packed sequence --> (b x sl x 2*lstm_dim)
        lstm_out,_ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)

        # Max pool along the last dimension
        ff_in = self.dropout(torch.max(lstm_out, 1)[0])
        # Some magic to get the last output of the BiLSTM for classification (b x 2*lstm_dim)
        #ff_in = lstm_out.gather(1, input_lens.view(-1,1,1).expand(lstm_out.size(0), 1, lstm_out.size(2)) - 1).squeeze()

        # Get logits (b x n_classes)
        logits = self.model['cls'](ff_in).view(-1, self.n_classes)
        outputs = (logits,)
        if labels is not None:
            # Xentropy loss
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
            outputs = (loss,) + outputs

        return outputs


In [ ]:
bpemb_en = BPEmb(lang='en', dim=100, vs=25000)

pretrained_embeddings = np.concatenate([bpemb_en.emb.vectors, np.zeros(shape=(1,100))], axis=0)


lstm_dim = 100

device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")

model = BiLSTMNetwork(
pretrained_embeddings=torch.FloatTensor(pretrained_embeddings),
lstm_dim=lstm_dim,
n_classes=2
).to(device)

In [ ]:
def accuracy(logits, labels):
  logits = np.asarray(logits).reshape(-1, len(logits[0]))
  labels = np.asarray(labels).reshape(-1)
  return np.sum(np.argmax(logits, axis=-1) == labels).astype(np.float32) / float(labels.shape[0])

def evaluate(model: nn.Module, valid_dl: DataLoader):
  """
  Evaluates the model on the given dataset
  :param model: The model under evaluation
  :param valid_dl: A `DataLoader` reading validation data
  :return: The accuracy of the model on the dataset
  """
  # VERY IMPORTANT: Put your model in "eval" mode -- this disables things like
  # layer normalization and dropout
  model.eval()
  labels_all = []
  logits_all = []

  # ALSO IMPORTANT: Don't accumulate gradients during this process
  with torch.no_grad():
    for batch in tqdm(valid_dl, desc='Evaluation'):
      batch = tuple(t.to(device) for t in batch)
      input_ids = batch[0]
      seq_lens = batch[1]
      labels = batch[2]

      _, logits = model(input_ids, seq_lens, labels=labels)
      labels_all.extend(list(labels.detach().cpu().numpy()))
      logits_all.extend(list(logits.detach().cpu().numpy()))
    acc = accuracy(logits_all, labels_all)

    return acc,labels_all,logits_all



In [ ]:
def train(
    model: nn.Module,
    train_dl: DataLoader,
    valid_dl: DataLoader,
    optimizer: torch.optim.Optimizer,
    n_epochs: int,
    device: torch.device,
    patience: int = 10
):
  """
  The main training loop which will optimize a given model on a given dataset
  :param model: The model being optimized
  :param train_dl: The training dataset
  :param valid_dl: A validation dataset
  :param optimizer: The optimizer used to update the model parameters
  :param n_epochs: Number of epochs to train for
  :param device: The device to train on
  :return: (model, losses) The best model and the losses per iteration
  """

  # Keep track of the loss and best accuracy
  losses = []
  best_acc = 0.0
  pcounter = 0

  # Iterate through epochs
  for ep in range(n_epochs):

    loss_epoch = []

    #Iterate through each batch in the dataloader
    for batch in tqdm(train_dl):
      # VERY IMPORTANT: Make sure the model is in training mode, which turns on
      # things like dropout and layer normalization
      model.train()

      # VERY IMPORTANT: zero out all of the gradients on each iteration -- PyTorch
      # keeps track of these dynamically in its computation graph so you need to explicitly
      # zero them out
      optimizer.zero_grad()

      # Place each tensor on the GPU
      batch = tuple(t.to(device) for t in batch)
      input_ids = batch[0]
      seq_lens = batch[1]
      labels = batch[2]

      # Pass the inputs through the model, get the current loss and logits
      loss, logits = model(input_ids, seq_lens, labels=labels)
      losses.append(loss.item())
      loss_epoch.append(loss.item())

      # Calculate all of the gradients and weight updates for the model
      loss.backward()

      # Optional: clip gradients
      #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

      # Finally, update the weights of the model
      optimizer.step()
      #gc.collect()

    # Perform inline evaluation at the end of the epoch
    acc,_,_ = evaluate(model, valid_dl)
    print(f'Validation accuracy: {acc}, train loss: {sum(loss_epoch) / len(loss_epoch)}')

    # Keep track of the best model based on the accuracy
    if acc > best_acc:
      torch.save(model.state_dict(), 'best_model')
      best_acc = acc
      pcounter = 0
    else:
      pcounter += 1
      if pcounter == patience:
        break
        #gc.collect()

  model.load_state_dict(torch.load('best_model'))
  return model, losses

In [ ]:
def flatten_ids(ids):
    # Flad alle niveauer af lister til én liste af ints
    while isinstance(ids, list) and len(ids) == 1 and isinstance(ids[0], list):
        ids = ids[0]
    return ids


In [ ]:
def collate_batch_bilstm(batch):
    input_ids = []
    labels = []
    seq_lens = []

    for item in batch:
        ids = flatten_ids(item[0])
        input_ids.append(ids)
        seq_lens.append(len(ids))
        labels.append(item[2])  # <-- brug item[2], ikke item[1]

    # Pad sequences
    max_len = max(seq_lens)
    input_ids_padded = [ids + [0]*(max_len - len(ids)) for ids in input_ids]

    return torch.tensor(input_ids_padded, dtype=torch.long), \
           torch.tensor(seq_lens, dtype=torch.long), \
           torch.tensor(labels, dtype=torch.long)


In [ ]:
def text_to_batch_bilstm(text: List, tokenizer, max_len=512) -> Tuple[List, List]:
    """
    Creates a tokenized batch for input to a bilstm model
    :param text: A list of sentences to tokenize
    :param tokenizer: A tokenization function to use (i.e. fasttext)
    :return: Tokenized text as well as the length of the input sequence
    """
    # Some light preprocessing
    input_ids = [tokenizer.encode_ids_with_eos(t)[:max_len] for t in text]

    return input_ids, [len(ids) for ids in input_ids]

In [ ]:
class ClassificationDatasetReader(Dataset):
    def __init__(self, df, tokenizer):
        self.df = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]  # bedre end values[idx], så vi kan bruge kolonnenavne
        input_ids, seq_lens = text_to_batch_bilstm([row['question']], self.tokenizer)

        # Remap True/False til 1/0
        label = 1 if row['answerable'] else 0

        return input_ids, seq_lens, label


In [ ]:
def evaluate(model: nn.Module, valid_dl: DataLoader):
    """
    Evaluates the model on the given dataset.
    Returns accuracy, precision, recall, f1, labels, and logits.
    """
    model.eval()
    labels_all, logits_all = [], []

    with torch.no_grad():
        for batch in tqdm(valid_dl, desc='Evaluation'):
            batch = tuple(t.to(device) for t in batch)
            input_ids, seq_lens, labels = batch

            logits = model(input_ids, seq_lens)

            labels_all.extend(labels.cpu().numpy())
            logits_all.extend(logits.cpu().numpy())

    labels_all = np.array(labels_all)
    preds = np.argmax(np.array(logits_all), axis=-1)

    acc = (preds == labels_all).mean()
    precision = precision_score(labels_all, preds, zero_division=0)
    recall = recall_score(labels_all, preds, zero_division=0)
    f1 = f1_score(labels_all, preds, zero_division=0)

    return acc, precision, recall, f1, labels_all, logits_all


In [ ]:
languages = ["ar", "te", "ko"]
batch_size = 32
lr = 3e-4
n_epochs = 100
num_workers = 2

optimizer = AdamW(model.parameters(), lr=lr)

for lang in languages:
    print(f"\n=== Training on {lang.upper()} ===")

    df_lang_train = df_train_clean[df_train_clean['lang'] == lang].reset_index(drop=True)
    df_lang_val = df_val_clean[df_val_clean['lang'] == lang].reset_index(drop=True)

    train_dataset = ClassificationDatasetReader(df_lang_train, bpemb_en)
    train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          collate_fn=collate_batch_bilstm, num_workers=num_workers)

    valid_dataset = ClassificationDatasetReader(df_lang_val, bpemb_en)
    valid_dl = DataLoader(valid_dataset, batch_size=len(df_lang_val),
                          collate_fn=collate_batch_bilstm, num_workers=num_workers)

    model, losses = train(model, train_dl, valid_dl, optimizer, n_epochs, device)


## DistilBERT

### Arabic

In [ ]:
label_encoder = preprocessing.LabelEncoder()
arabicDf_train_clean['label'] = arabicDf_train_clean['answerable'].astype(int)
arabicDf_val_clean['label'] = arabicDf_val_clean['answerable'].astype(int)

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize_data(df, tokenizer):
    return tokenizer(
        df['question'].tolist(),
        df['context'].tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

train_dataset_ar = Dataset.from_pandas(arabicDf_train_clean[['question', 'context', 'label']])
val_dataset_ar = Dataset.from_pandas(arabicDf_val_clean[['question', 'context', 'label']])

def preprocess(examples):
    tokenized = tokenizer(examples['question'], examples['context'], truncation=True, padding='max_length')
    tokenized['labels'] = examples['label']
    return tokenized

train_dataset_ar = train_dataset_ar.map(preprocess, batched=True)
val_dataset_ar = val_dataset_ar.map(preprocess, batched=True)

arabicDf_train_clean

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_ar,
    eval_dataset=val_dataset_ar,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model("./arabic_bert_model")


In [ ]:
# Run evaluation
metrics = trainer.evaluate()
print("Validation metrics:", metrics)

trainer.evaluate()

predictions = trainer.predict(val_dataset_ar)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Show first 20 comparisons
for i in range(100):
    print(f"Predicted: {preds[i]}, Actual: {labels[i]}")


## Telugu

In [ ]:
label_encoder = preprocessing.LabelEncoder()
teluguDf_train_clean['label'] = teluguDf_train_clean['answerable'].astype(int)
teluguDf_val_clean['label'] = teluguDf_val_clean['answerable'].astype(int)

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize_data(df, tokenizer):
    return tokenizer(
        df['question'].tolist(),
        df['context'].tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

train_dataset_te = Dataset.from_pandas(teluguDf_train_clean[['question', 'context', 'label']])
val_dataset_te = Dataset.from_pandas(teluguDf_val_clean[['question', 'context', 'label']])

def preprocess(examples):
    tokenized = tokenizer(examples['question'], examples['context'], truncation=True, padding='max_length')
    tokenized['labels'] = examples['label']
    return tokenized

train_dataset_te = train_dataset_te.map(preprocess, batched=True)
val_dataset_te = val_dataset_te.map(preprocess, batched=True)

teluguDf_train_clean

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_te = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_te,
    eval_dataset=val_dataset_te,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_te.train()

trainer_te.save_model("./telugu_bert_model")


In [ ]:
# Run evaluation
metrics = trainer_te.evaluate()
print("Validation metrics:", metrics)

trainer_te.evaluate()

predictions = trainer_te.predict(val_dataset_te)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Show first 20 comparisons
for i in range(360):
    print(f"Predicted: {preds[i]}, Actual: {labels[i]}")

## Korean

In [ ]:
label_encoder = preprocessing.LabelEncoder()
koreanDf_train_clean['label'] = koreanDf_train_clean['answerable'].astype(int)
koreanDf_val_clean['label'] = koreanDf_val_clean['answerable'].astype(int)

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize_data(df, tokenizer):
    return tokenizer(
        df['question'].tolist(),
        df['context'].tolist(),
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

train_dataset_ko = Dataset.from_pandas(koreanDf_train_clean[['question', 'context', 'label']])
val_dataset_ko = Dataset.from_pandas(koreanDf_val_clean[['question', 'context', 'label']])

def preprocess(examples):
    tokenized = tokenizer(examples['question'], examples['context'], truncation=True, padding='max_length')
    tokenized['labels'] = examples['label']
    return tokenized

train_dataset_ko = train_dataset_ko.map(preprocess, batched=True)
val_dataset_ko = val_dataset_ko.map(preprocess, batched=True)

koreanDf_train_clean

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_ko = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_ko,
    eval_dataset=val_dataset_ko,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_ko.train()

trainer_ko.save_model("./korean_bert_model")


In [ ]:
# Run evaluation
metrics = trainer_ko.evaluate()
print("Validation metrics:", metrics)

trainer_ko.evaluate()

predictions = trainer_ko.predict(val_dataset_ko)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Show first 20 comparisons
for i in range(20):
    print(f"Predicted: {preds[i]}, Actual: {labels[i]}")
